# Recommender evaluation & bias — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def cosine(a,b):
    a=np.asarray(a,dtype=float); b=np.asarray(b,dtype=float)
    return float(a@b/(np.linalg.norm(a)*np.linalg.norm(b)))
def sigmoid(x): return 1/(1+np.exp(-np.asarray(x)))
def dcg(rels): return sum((2**r-1)/math.log2(i+2) for i,r in enumerate(rels))
def ndcg(rels):
    best=sorted(rels, reverse=True)
    return dcg(rels)/dcg(best) if dcg(best)>0 else 0.0
print('setup ok')

## Estimating ranking quality despite logging, exposure, and feedback bias
Evaluation is hard because recommenders shape the data they later learn from. The examples compute offline holdout metrics, exposure bias, inverse-propensity correction, position bias, and feedback-loop concentration.

In [ ]:
# 1) holdout recall@3 measures whether hidden positives are retrieved
scores=np.array([[.9,.8,.1,.2],[.1,.7,.6,.5],[.4,.3,.9,.8]]); true=[{0,1},{2},{3}]
top=np.argsort(-scores,axis=1)[:,:3]; rec=np.mean([len(set(top[i])&true[i])/len(true[i]) for i in range(3)])
plt.figure(figsize=(6,3)); plt.bar(['u0','u1','u2'],[len(set(top[i])&true[i])/len(true[i]) for i in range(3)]); plt.title(f'recall@3 = {rec:.3f}')
assert abs(rec-1.0)<1e-12

In [ ]:
# 2) exposure bias: unshown items have no feedback, not necessarily no relevance
relevance=np.array([1,1,0,1]); exposed=np.array([1,0,1,0]); observed_click=relevance*exposed
plt.figure(figsize=(6,3)); plt.bar(['true relevant','observed positive'],[relevance.sum(),observed_click.sum()]); plt.title('missing exposure hides positives')
assert relevance.sum()==3 and observed_click.sum()==1

In [ ]:
# 3) inverse propensity scoring reweights scarce exposures
click=np.array([1,0,1]); prop=np.array([0.5,0.2,0.1]); ips=float(np.mean(click/prop)); naive=float(click.mean())
plt.figure(figsize=(6,3)); plt.bar(['naive','IPS'],[naive,ips]); plt.title('IPS corrects logging propensities')
assert abs(ips-4.0)<1e-12 and ips>naive

In [ ]:
# 4) position bias makes top slots look better even at equal relevance
exam=np.array([1.0,.6,.3,.15]); rel=np.ones(4)*.5; observed=exam*rel
plt.figure(figsize=(6,3)); plt.bar(['rank1','rank2','rank3','rank4'],observed); plt.title('click rate = relevance x examination')
assert observed[0]>observed[-1] and abs(observed[-1]-0.075)<1e-12

In [ ]:
# 5) feedback loop: repeated exposure concentrates traffic on the current winner
traffic=np.array([1/3,1/3,1/3]); ctr=np.array([.05,.06,.055]); history=[]
for _ in range(10):
    traffic=0.8*traffic+0.2*(ctr*traffic)/(ctr@traffic); history.append(traffic.copy())
history=np.array(history)
plt.figure(figsize=(6,3)); plt.plot(history); plt.title('traffic concentrates through feedback'); plt.legend(['A','B','C'])
assert history[-1,1]>history[0,1]